# ParkViewRT YOLO Model Comparison

This notebook compares pretrained YOLO models on the current FCI and FAIE parking images.

Models compared by default:

- `yolov8n.pt`
- `yolo11n.pt`
- `yolo26n.pt`

The notebook uses the same slot-polygon overlap idea as the FastAPI backend. It exports CSV files and annotated debug images for FYP2 evaluation and model-selection evidence, not custom training.

In [1]:
!pip install -U ultralytics shapely opencv-python-headless pandas

   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   -------------------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Project Path

Upload or clone the project into Colab, then set `PROJECT_ROOT` to the repository path.

For example, if the repo is uploaded to `/content/parkviewrt`, keep the default below.

In [2]:
from pathlib import Path

PROJECT_CANDIDATES = [Path.cwd(), Path.cwd().parent, Path('/content/parkviewrt')]
PROJECT_ROOT = next((path for path in PROJECT_CANDIDATES if (path / 'backend' / 'app').exists()), PROJECT_CANDIDATES[-1])
APP_DIR = PROJECT_ROOT / 'backend' / 'app'
IMAGES_DIR = APP_DIR / 'data' / 'images'
SLOTS_DIR = APP_DIR / 'data' / 'slots'
OUTPUT_DIR = PROJECT_ROOT / 'colab' / 'outputs'
DEBUG_DIR = OUTPUT_DIR / 'debug_images'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEBUG_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Images dir:', IMAGES_DIR)
print('Slots dir:', SLOTS_DIR)

required_paths = [IMAGES_DIR, SLOTS_DIR]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f'Missing project folders: {missing_paths}')

Project root: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt
Images dir: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt\backend\app\data\images
Slots dir: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt\backend\app\data\slots


## Evaluation Configuration

The current static images have all labelled slots occupied. When you obtain new images or real video frames, update `EXPECTED_OCCUPANCY` to match the visible ground truth.

In [3]:
MODELS = ['yolov8n.pt', 'yolo11n.pt', 'yolo26n.pt']
LOCATIONS = ['fci', 'faie']

CONFIDENCE = 0.20
IMAGE_SIZE = 1600
SLOT_THRESHOLD = 0.30
BOX_THRESHOLD = 0.20
RUNS_PER_MODEL = 3

EXPECTED_OCCUPANCY = {
    'fci': {slot_id: True for slot_id in ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8']},
    'faie': {slot_id: True for slot_id in ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8']},
}

In [4]:
import json
import time

import cv2
import numpy as np
import pandas as pd
from shapely.geometry import Point, Polygon, box
from ultralytics import YOLO


def load_slots(location_id):
    with open(SLOTS_DIR / f'{location_id}_slots.json', 'r', encoding='utf-8') as file:
        return json.load(file)['slots']


def image_path_for(location_id):
    for extension in ('.jpg', '.jpeg', '.png'):
        path = IMAGES_DIR / f'{location_id}{extension}'
        if path.exists():
            return path
    raise FileNotFoundError(f'No image found for {location_id}')


def detect_cars(model, image_path):
    elapsed_times = []
    results = None

    for _ in range(RUNS_PER_MODEL):
        start = time.perf_counter()
        results = model(str(image_path), conf=CONFIDENCE, imgsz=IMAGE_SIZE, verbose=False)
        elapsed_times.append((time.perf_counter() - start) * 1000)

    elapsed_ms = sum(elapsed_times) / len(elapsed_times)
    detections = []

    for result in results:
        for detected_box in result.boxes:
            class_name = result.names[int(detected_box.cls[0].item())]
            if class_name != 'car':
                continue
            x1, y1, x2, y2 = detected_box.xyxy[0].tolist()
            detections.append({
                'confidence': float(detected_box.conf[0].item()),
                'bbox': [float(x1), float(y1), float(x2), float(y2)],
            })

    return detections, elapsed_ms


def calculate_occupancy(slots, detections):
    results = []

    for slot in slots:
        slot_polygon = Polygon(slot['points'])
        slot_area = slot_polygon.area
        occupied = False
        best_slot_overlap = 0.0
        best_box_overlap = 0.0

        if slot_area > 0:
            for detection in detections:
                detection_box = box(*detection['bbox'])
                detection_area = detection_box.area
                intersection_area = slot_polygon.intersection(detection_box).area
                slot_overlap = intersection_area / slot_area
                box_overlap = intersection_area / detection_area if detection_area else 0
                detection_center = Point(detection_box.centroid.x, detection_box.centroid.y)

                best_slot_overlap = max(best_slot_overlap, slot_overlap)
                best_box_overlap = max(best_box_overlap, box_overlap)

                if (
                    slot_overlap >= SLOT_THRESHOLD
                    or box_overlap >= BOX_THRESHOLD
                    or slot_polygon.contains(detection_center)
                    or detection_box.contains(slot_polygon.centroid)
                ):
                    occupied = True

        results.append({
            'slot_id': slot['slot_id'],
            'occupied': occupied,
            'slot_overlap': best_slot_overlap,
            'box_overlap': best_box_overlap,
        })

    return results


def score_predictions(location_id, predictions):
    expected = EXPECTED_OCCUPANCY[location_id]
    correct = 0
    total = len(predictions)

    for prediction in predictions:
        if expected.get(prediction['slot_id']) == prediction['occupied']:
            correct += 1

    return correct / total if total else 0


def draw_debug_image(image_path, slots, detections, occupancy, output_path):
    image = cv2.imread(str(image_path))
    if image is None:
        raise FileNotFoundError(f'Unable to read image: {image_path}')

    occupancy_by_slot = {slot['slot_id']: slot for slot in occupancy}

    for detection in detections:
        x1, y1, x2, y2 = [int(value) for value in detection['bbox']]
        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 170, 0), 2)
        cv2.putText(image, f"car {detection['confidence']:.2f}", (x1, max(20, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 170, 0), 2)

    for slot in slots:
        predicted = occupancy_by_slot[slot['slot_id']]
        color = (0, 0, 255) if predicted['occupied'] else (0, 180, 0)
        polygon_points = np.array(slot['points'], dtype=np.int32)
        cv2.polylines(image, [polygon_points], isClosed=True, color=color, thickness=3)
        centroid = polygon_points.mean(axis=0).astype(int)
        label = f"{slot['slot_id']} {'occ' if predicted['occupied'] else 'free'}"
        cv2.putText(image, label, tuple(centroid), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    cv2.imwrite(str(output_path), image)

## Run Model Comparison

In [5]:
summary_rows = []
slot_rows = []

for model_name in MODELS:
    print(f'Loading {model_name}...')
    model = YOLO(model_name)

    for location_id in LOCATIONS:
        slots = load_slots(location_id)
        image_path = image_path_for(location_id)
        detections, elapsed_ms = detect_cars(model, image_path)
        occupancy = calculate_occupancy(slots, detections)
        accuracy = score_predictions(location_id, occupancy)
        occupied_count = sum(1 for slot in occupancy if slot['occupied'])
        safe_model_name = model_name.replace('.pt', '').replace('/', '_')
        debug_image_path = DEBUG_DIR / f'debug_{safe_model_name}_{location_id}.jpg'
        draw_debug_image(image_path, slots, detections, occupancy, debug_image_path)

        summary_rows.append({
            'model': model_name,
            'location_id': location_id,
            'detections': len(detections),
            'total_slots': len(occupancy),
            'occupied_count': occupied_count,
            'available_count': len(occupancy) - occupied_count,
            'slot_accuracy': accuracy,
            'inference_ms': elapsed_ms,
            'debug_image': str(debug_image_path),
        })

        for slot in occupancy:
            slot_rows.append({
                'model': model_name,
                'location_id': location_id,
                **slot,
            })

summary_df = pd.DataFrame(summary_rows)
slot_df = pd.DataFrame(slot_rows)

summary_df

Loading yolov8n.pt...
Loading yolo11n.pt...
Loading yolo26n.pt...


,model,location_id,detections,total_slots,occupied_count,available_count,slot_accuracy,inference_ms,debug_image
0,yolov8n.pt,fci,92,8,8,0,1.0,3133.727700,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
1,yolov8n.pt,faie,32,8,8,0,1.0,8443.494567,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
2,yolo11n.pt,fci,93,8,8,0,1.0,8107.660900,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
3,yolo11n.pt,faie,45,8,8,0,1.0,3805.919933,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
4,yolo26n.pt,fci,91,8,8,0,1.0,2655.649733,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
5,yolo26n.pt,faie,39,8,8,0,1.0,2551.305133,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...


In [6]:
summary_path = OUTPUT_DIR / 'model_comparison_summary.csv'
slot_path = OUTPUT_DIR / 'model_comparison_slots.csv'

summary_df.to_csv(summary_path, index=False)
slot_df.to_csv(slot_path, index=False)

print('Saved:', summary_path)
print('Saved:', slot_path)
print('Saved debug images in:', DEBUG_DIR)

summary_df.sort_values(['slot_accuracy', 'inference_ms'], ascending=[False, True])

Saved: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt\colab\outputs\model_comparison_summary.csv
Saved: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt\colab\outputs\model_comparison_slots.csv
Saved debug images in: d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimester\FYP2\parkviewrt\colab\outputs\debug_images


,model,location_id,detections,total_slots,occupied_count,available_count,slot_accuracy,inference_ms,debug_image
5,yolo26n.pt,faie,39,8,8,0,1.0,2551.305133,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
4,yolo26n.pt,fci,91,8,8,0,1.0,2655.649733,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
0,yolov8n.pt,fci,92,8,8,0,1.0,3133.727700,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
3,yolo11n.pt,faie,45,8,8,0,1.0,3805.919933,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
2,yolo11n.pt,fci,93,8,8,0,1.0,8107.660900,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...
1,yolov8n.pt,faie,32,8,8,0,1.0,8443.494567,d:\Malaysia\MMU_Studies\Year 3\2nd Long Trimes...


## Notes for FYP2 Report

Use the summary table to justify the selected pretrained model. Accuracy alone is not enough; inference time is also important for real-time or near-real-time monitoring. If YOLO26 is slower but not more accurate on the MMU images, YOLO11 or YOLOv8 may still be a better engineering choice.

Use the annotated debug images to explain false negatives and false positives. This is especially important because the current images are high-angle parking views where tree occlusion, small cars, and perspective distortion can affect pretrained model performance.